# Ligand-based screening: compound similarity

## Aim
We will perform a virtual screening in the form of a similarity search for the EGFR inhibitor Gefitinib against our dataset of EGFR-tested molecules from the ChEMBL database filtered by Lipinski's rule of five.

## For more details
https://github.com/volkamerlab/teachopencadd/blob/master/teachopencadd/talktorials/T004_compound_similarity/talktorial.ipynb

## Instructions
Replace XXX with the appropriate code

## Configuration

In [ ]:
# Imports
# 1. Standard library imports
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules

# 2. Third-party library imports
import matplotlib.pyplot as plt
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Draw,
    MACCSkeys,
    PandasTools,
    rdFingerprintGenerator,
)
PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f'{HERE}')
ROOT = HERE.parent
print(f'{ROOT}')
DATA = ROOT / 'data'
print(f'{DATA}')

## Virtual screening using similarity search

### Data

#### EGFR compounds

In [ ]:
# File to load
EGFR_compounds_lipinski_csv_path = DATA / "EGFR_compounds_lipinski.csv"

In [ ]:
# load data restricted to ['molecule_chembl_id', 'smiles', 'pIC50'] columns in a dataframe called egfr_df 
egfr_df = XXX.XXX(
    EGFR_compounds_lipinski_csv_path,
    usecols=XXX
    )

In [ ]:
# Add a molecule column with PandasTools
PandasTools.XXX(
    egfr_df,
    smilesCol='smiles'
)

In [ ]:
# check
print("Number of compounds: {egfr_df.shape[0]}")
egfr_df.head(2)

#### Gefitinib

In [ ]:
# Retrieve Gefitinib smiles from http://www.icoa.fr/pkidb/
gefitinib_smiles = "COc1cc2c(cc1OCCCN3CCOCC3)c(ncn2)Nc4ccc(c(c4)Cl)F"

# Convert smiles to mol
gefitinib_mol = XXX.XXX(gefitinib_smiles)
gefitinib_mol

## Fingerprints

In [ ]:
# Generate MACCS fingerprint for gefitinib
gefitinib_maccs_fp = MACCSkeys.XXX(gefitinib_mol)

In [ ]:
# Generate morgan fingerprint for gefitinib
fpg = rdFingerprintGenerator.XXX(radius=2, fpSize=2048)

gefitinib_morgan_fp = fpg.GetCountFingerprint(XXX)

In [ ]:
# Generate a list of MACCS fingerprints for all the egfr compounds
maccs_fps_l = egfr_df['ROMol'].map(MACCSkeys.GenMACCSKeys).XXX

In [ ]:
# Generate a list of Morgan fingerprints for all the egfr compounds
morgan_fps_l = egfr_df['ROMol'].map(fpg.GetCountFingerprint).XXX

## Similarities

In [ ]:
# Add the Tanimoto similarity between Gefitinib and egfr compounds using MACCS fingerprints to egfr_df
egfr_df['tanimoto_maccs'] = DataStructs.XXX(gefitinib_maccs_fp, maccs_fps_l)

In [ ]:
# Add the Tanimoto similarity between Gefitinib and egfr compounds using Morgan fingerprints to egfr_df
egfr_df['tanimoto_morgan'] = DataStructs.XXX(gefitinib_morgan_fp, morgan_fps_l)

In [ ]:
# Check egfr_df
print(f'egfr_df shape: {egfr_df.shape}')
egfr_df.head(2)

## Distribution of similarity values

In [ ]:
# Plot tanimoto_maccs and tanimoto_morgan distributions
# Define a figure with 2 axes
fig, axes = plt.subplots(figsize= (10, 4), nrows=1, ncols=2)

# add tanimoto_maccs histogram on axe[0,0]
egfr_df.hist([XXX], ax=axes[0])

# add tanimoto_morgan histogram on axe[0,1]
egfr_df.hist([XXX], ax=axes[1])


axes[0].set_ylim(0,3000)
axes[1].set_ylim(0,3000)

## Visualise most similar compounds to gefitinib

In [ ]:
# Display gefinib molecule
XXX

In [ ]:
# Sort egfr_df by tanimoto_morgan, descending order and show the first n rows
n = 5
egfr_df.XXX(
    XXX='tanimoto_morgan',
    ascending=XXX,
).XXX

In [ ]:
# Show on a MolsToGridImage the n most similar molecules (regarding tanimoto_morgan) and add #index molecule_chembl_id and pIC50 as legends.

# number of compounds to display next gefitinib
top_n_mols = 11

# sort molecules regarding their tanimoto_morgan values and reset_index
top_mols_df = egfr_df.XXX(by=XXX, ascending=False).reset_index()

# select top_n_mols compounds
top_mols = top_mols_df[XXX]

# define legends
legends = [
    f'#{index+1} {row['molecule_chembl_id']}, pIC50={row['pIC50']:.2f}' for index, row in top_mols.iterrows()
]

# Image with gefitinib added
Draw.XXX(
    XXX=[gefitinib_mol] + top_mols['ROMol'].tolist(),
    legends=(['Gefitinib'] + legends),
    molsPerRow=3,
    subImgSize=(450, 350)
)

In order to check how well the similarity search is able to distinguish between active and inactive molecules based on our dataset we need to use the bioactivity values and generate enrichment plots to see the ratio of detected active molecules.

See TeachOpenCADD T004_compound_similarity for more details.